# TM-RugPull dataset initial analysis

## Data collection

In [ ]:
# Loading data from .xlsx file

import pandas as pd
import numpy as np
import matplotlib.pyplot as pyplot

file = 'data/TM-RugPull.xlsx'

data = pd.read_excel(file)

# Remove a space in the end of some column names
data.columns = data.columns.str.strip()

print(data.shape)

pd.set_option('display.max_columns', None)

data.head(5)

In [ ]:
# Check the balance between classes in the whole dataset
# Source: https://note.nkmk.me/en/python-pandas-value-counts/#value_counts

class_counts = data['class'].value_counts()
class_percent = data['class'].value_counts(normalize=True) * 100

print("Counts:", class_counts.to_string(), "\nIn %:", class_percent.to_string())

In [ ]:
# Check general info about data

print("\nDataset information:")
data.info()

In [ ]:
# Delete columns that are not useful for further analysis

data.drop(columns=['Project Title', 'Sign', 'website', 'x profile', 'Smart Contract (online)', 'smart Contract (offline)', 'project starting date', 'project end date'], inplace=True)

# Check the transformation
data.head(10)

In [ ]:
# Analysis of blockchain distribution in data

blockchain_distribution = data['Blockchain'].value_counts(normalize=True) * 100
print(blockchain_distribution.to_string())

In [ ]:
# Analyse scam rate per chain

scam_rate_per_chain = (
    data.groupby('Blockchain')['class']
        .apply(lambda s: (s == 'scam').mean())
        .sort_values(ascending=False)
)

scam_rate_percents = (scam_rate_per_chain * 100)

print(scam_rate_percents.to_string())

##### As far as FANTOM, CRONO, BASE and SNOW blockchains have very few samples (less than 10) and are biased (only scam tokens for BASE, CRONO and FANTOM and only normal samples for SNOW), these blockchains will be removed from the dataset to avoid overfitting.

In [ ]:
# Drop blockchains with number of samples less than 1%

min_samples = 1.0
chains_to_keep = blockchain_distribution[blockchain_distribution >= min_samples].index

print("Blockchains to keep:")
print(list(chains_to_keep))

print("\nDropped blockchains:")
print(blockchain_distribution[blockchain_distribution < min_samples].to_string())

data = data[data['Blockchain'].isin(chains_to_keep)].copy()


In [ ]:
# Dataset after dropping blockchains with small number of samples

print(data.shape)
data.head(5)

In [ ]:
# Display blockchain distribution after dropping blockchains with small number of samples

blockchain_distribution = data['Blockchain'].value_counts(normalize=True) * 100
print(blockchain_distribution.to_string())

##### Create a test set and a validation test

In [ ]:
# Create a test set and a validation set from the raw data to avoid data leakage
# Validation set size is about 10,5% and test set size is about 24,5% from the whole data set
# TODO reference to AML tutorial

from sklearn.model_selection import train_test_split

# define size of data both for test and validatioin sets, define the seed for all subsequent experiments
test_and_val_size = 0.35
seed = 7

# Split the data first on train set and set for test and validation
train_set, test_and_val_set = train_test_split(data, test_size=test_and_val_size, random_state=seed, stratify=data['class'])

# Split the part for test and validation into test set and validation set
test_set, val_set = train_test_split(test_and_val_set, test_size=0.3, random_state=seed, stratify=test_and_val_set['class'])

# Output the shapes to verify the splits
print("Training set shape:", train_set.shape)
print("Test set shape:", test_set.shape)
print("Validation set shape:", val_set.shape)

# Create a list of sets to perform further feature engeneering on all subsets of data
data_sets = [train_set, test_set, val_set]

In [ ]:
# Verify that there is no overlap between sets, no data leak at this stage
print("Overlap between train and test:", np.intersect1d(train_set.index, test_set.index).size)
print("Overlap between train and validation:", np.intersect1d(train_set.index, val_set.index).size)
print("Overlap between test and validation:", np.intersect1d(test_set.index, val_set.index).size)

## Raw data analysis

#### Performed first on raw data to get overall idea of what the dataset contains
##### All further exploratory operations except checking general info should be done only on the train set to avoid data leakage.
##### Analysis of skew and correlation is performed only on numeric features first. Categorial features are initially analysed separately, then encoded during pre-processing.

In [ ]:
# Encode class label

from sklearn.preprocessing import LabelEncoder

for set in data_sets:
    le = LabelEncoder()
    set['class'] = set['class'].map({'normal': 0, 'scam': 1})

train_set

In [ ]:
# Check the data description

description = train_set.describe()
description

In [ ]:
# Check the balance between classes in the training set

class_counts_train = data['class'].value_counts()
class_percent_train = data['class'].value_counts(normalize=True) * 100

print("Counts:", class_counts_train.to_string(), "\nIn %:", class_percent_train.to_string())

In [ ]:
print(train_set.dtypes)

In [ ]:
# Transfer all data to numeric values

#TODO: reference from notes

# Columns with object datatype
cols_to_clean = [
    'the number of Transactions',
    'Token concentration ratio per holder',
    'Token balance',
    'first deposits',
    'Google results for project website (first day)',
    'Google results for project website (duration/2)'
]

# Do it for all sets
data_sets = [train_set, test_set, val_set]

for i, d_set in enumerate(data_sets):
    for col in cols_to_clean:
        data_sets[i][col] = pd.to_numeric(
            d_set[col].astype(str)
                   .str.replace('\xa0', '', regex=False)        # Remove non-breaking whitespaces
                   .str.replace(',', '', regex=False)           # Strip comas separating numeric values
                   .str.strip(),                                # Remove surrounding whitespaces
            errors='coerce'                                     # If cannot parse, put NaN
        )

train_set, test_set, val_set = data_sets

# Verify
print(train_set[cols_to_clean].dtypes)
print(train_set[cols_to_clean].isnull().sum())
print(train_set[cols_to_clean].describe())

In [ ]:
# Check for skew for numeric columns

numeric_cols = train_set.select_dtypes(include=['number']).columns
skew = train_set[numeric_cols].skew()

skew

### Visualisation

##### Based on CSM010-2024-APR, Topic 2, Lab.2.15. Analysing data

In [ ]:
# Histograms

train_set.hist(figsize=[30, 30])
pyplot.show()

In [ ]:
# Density plots

train_set.plot(kind='density', subplots=True, layout=(8,7), sharex=False, sharey=False, figsize=[30, 30])
pyplot.show()

In [ ]:
# Box and Whisker Plots

train_set.plot(kind='box', subplots=True, layout=(8,7), sharex=False, sharey=False,figsize=[30, 30])
pyplot.show()

##### Data is extremely skewed and there are extreme outliers, thus, data requires pre-processing.

In [ ]:
# Search for correlations of numeric features

correlations = train_set[numeric_cols].corr(method='pearson')

correlations

In [ ]:
# Correlation Matrix Plot

fig = pyplot.figure(figsize=[30, 30])
ax = fig.add_subplot(111)
cax = ax.matshow(correlations, vmin=-1, vmax=1)
fig.colorbar(cax)

ticks = np.arange(len(correlations.columns))

ax.set_xticks(ticks)
ax.set_yticks(ticks)

short_names = list(train_set[numeric_cols].columns)

ax.set_xticklabels(short_names)
ax.set_yticklabels(short_names)

pyplot.xticks(rotation=90)     # https://www.geeksforgeeks.org/how-to-rotate-x-axis-tick-label-text-in-matplotlib/?ysclid=lxdkiwmkrh456759424

pyplot.show()

In [ ]:
# Scatter plot Matrix
# TODO: make it more readable

pd.plotting.scatter_matrix(data, figsize=[20, 20])

pyplot.show()

##### Correlation matrix shows strong correlation (almost 1.0) between 'first deposits' and 'Q1'. Manual analysis reveal that 'first deposits' column is likely to be corrupted, since a lot of samples have the same float value here as in 'Q1', which does not make sense (there should be an integer).
##### There is also a strong correlation between price features (Q1, Q2, Q3 and Q4), thus, it is considered to drop three of them.

## General pre-processing of data

#### Based on results of data analysis on raw data, some pre-processing is required. In this section, general pre-processing will be applied (handling missing values, heavy skew, outliers), then both raw version and pre-processed version will be saved for further experimentation.

#### Data after general pre-processing also will be used for plotting to assess the effect of pre-processing on data.

In [ ]:
# Remove 'first deposits' manually, since it seems to be corrupted (in majority cases it reproduces Q1 figures, which are prices, not deposits count)
# TODO: consider to replace 'first deposits' (number of initial liquidity deposit events)??

for data_set in data_sets:
    data_set.drop(columns=['first deposits'], inplace=True)

train_set

In [ ]:
# Handle missing values (for numeric 'median' is used due to heavy skew, for categorical values 'most frequent' is used)
# SimpleImputer is trained for all numeric and categorical columns (not only for ones with missing values) to be able to handle possible future missing values.

# Based on: Géron, A. "End-to-end Machine Learning Project" in 'Hands-on' machine learning with Scikit-Learn, Keras & Tensorflow. (O'Reilly Media, Inc, 2019) 2nd edition.
# TODO: reference to docs

from sklearn.impute import SimpleImputer

# Define numeric and categorical columns
numeric_cols = train_set.select_dtypes(include=['number']).columns.tolist()
categorical_cols = train_set.select_dtypes(exclude=['number']).columns.tolist()

# Create inputers and fit to train set
num_imputer = SimpleImputer(strategy='median')
cat_imputer = SimpleImputer(strategy='most_frequent')
num_imputer.fit(train_set[numeric_cols])
cat_imputer.fit(train_set[categorical_cols])

# Apply to train, test, val sets
for data_set in [train_set, test_set, val_set]:
    data_set[numeric_cols] = num_imputer.transform(data_set[numeric_cols])
    data_set[categorical_cols] = cat_imputer.transform(data_set[categorical_cols])

# Verify it works (no missing values left)
print("\nRemaining missing values per column in train_set:")
print(train_set.isnull().sum())

In [ ]:
# Determine unique categories for 'Blockchain' and 'Blockchain Type' columns

print(data_sets[0]['Blockchain'].unique())
print(data_sets[0]['Blockchain Type'].unique())

In [ ]:
# Encode categorical features ('Blockchain', 'Blockchain Type') to make all features numeric.
# OneHotEncoder was chosen, since the number of categories is small (4 for 'Blockchain' and 3 for 'Blockchain Type',
# and OneHotEncoder prevents algorithms to treat close values more similar than distant ones.

# Based on: Based on: Géron, A. "End-to-end Machine Learning Project" in 'Hands-on' machine learning with Scikit-Learn, Keras & Tensorflow. (O'Reilly Media, Inc, 2019) 2nd edition.
# TODO: Reference to docs OneHotEncoder

from sklearn.preprocessing import OneHotEncoder

categorical_cols = ['Blockchain', 'Blockchain Type']
cat_encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
cat_encoder.fit(train_set[categorical_cols])

# Apply to each split
for data_set in [train_set, val_set, test_set]:
    cat_array = cat_encoder.transform(data_set[categorical_cols])
    cat_feature_names = cat_encoder.get_feature_names_out(categorical_cols)
    cat_df = pd.DataFrame(cat_array, columns=cat_feature_names, index=data_set.index)

    # Drop original categorical columns
    data_set.drop(columns=categorical_cols, inplace=True)

    # Add encoded columns
    for col in cat_df.columns:
        data_set[col] = cat_df[col]

# Verify it works (all columns are numeric)
print(train_set.dtypes)

In [ ]:
#

In [ ]:
# TODO: data require preparation and pre-processing (heavy skew, issues with 'first deposit' column (it seems that it is incorrect, duplicating Q1), outliers + a couple of missing values

# TODO: preparation and pre-processing

## Model specific pre-processing

#### Models from different families will be trained on data. Some of them require specific pre-processing. In this section, model specific pre-processing will be handled and then all versions of data (raw, after general pre-processing and after specific pre-processing will be used to train models and evaluate performance).